# 🚀 Mini Project — Pipeline Penjualan Q2 + Data Pelanggan
**Waktu:** ±60 menit (berkelompok 2–3 orang) · **Presentasi:** 3 menit/kelompok

## Pertanyaan bisnis
> *Segmen pelanggan mana yang paling menguntungkan di Q2 2026, bagaimana trennya per bulan, dan siapa pelanggan terbesar?*

## Sumber data
| Data | API | CSV cadangan |
|---|---|---|
| Transaksi Q2 | `api/static/sales_q2_page_{page}.json` | `data/sales_q2_raw.csv` |
| Pelanggan | `api/static/customers.json` | `data/customers_raw.csv` |

## Penilaian
| Aspek | Bobot |
|---|---|
| Pipeline berjalan end-to-end (Extract → Load) | 30% |
| Kualitas data cleaning (lolos semua cek) | 25% |
| Ketepatan query SQL | 20% |
| Dashboard & insight | 25% |

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [171]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

Konfigurasi OK — login database sebagai peserta25 @ ep-wandering-math-b3bml8w1-pooler.c-4.ap-southeast-1.aws.neon.tech


In [172]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

In [173]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

('peserta25', 'peserta25')


In [174]:
def run_sql(query):
    """Menjalankan query SELECT dan mengembalikan DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [175]:
if "{page}" in API_URL:   # mode static JSON
    API_Q2 = API_URL.replace("sales_page_{page}", "sales_q2_page_{page}")
    API_CUSTOMERS = API_URL.replace("sales_page_{page}", "customers")
else:                     # mode API server
    API_Q2 = API_URL.rsplit("/sales", 1)[0] + "/sales-q2"
    API_CUSTOMERS = API_URL.rsplit("/sales", 1)[0] + "/customers"
print(API_Q2, API_CUSTOMERS, sep="\n")

https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_q2_page_{page}.json
https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/customers.json


## Tahap 1 — Extract
Ambil transaksi Q2 (berhalaman) dan data pelanggan (satu respons). Gunakan fallback CSV jika perlu.

In [176]:
sales_data = []
page = 1

while True:
    response = requests.get(page_url(API_Q2, page))
    response.raise_for_status()
    result = response.json()

    sales_data.extend(result["data"])

    if not result.get("has_next", False):
        break

    page += 1

df_sales_raw = pd.DataFrame(sales_data)

print("Jumlah transaksi mentah:", len(df_sales_raw))

Jumlah transaksi mentah: 264


In [177]:
print("RAW_BASE:", repr(RAW_BASE))
print("API_URL:", repr(API_URL))
print("URL:", repr(URL))
print("page_url:", repr(page_url))
print("API_Q2:", repr(API_Q2))
print("API_CUSTOMERS:", repr(API_CUSTOMERS))

RAW_BASE: 'https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main'
API_URL: 'https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_page_{page}.json'
URL: <class 'sqlalchemy.engine.url.URL'>
page_url: <function page_url at 0x000001A4053CE820>
API_Q2: 'https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_q2_page_{page}.json'
API_CUSTOMERS: 'https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/customers.json'


In [178]:
import inspect

print("API_URL:", API_URL)
print("\nFungsi page_url:")
print(inspect.getsource(page_url))

API_URL: https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_page_{page}.json

Fungsi page_url:
def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"



In [179]:
# Tes URL transaksi bawaan notebook
print("API_URL lengkap:")
print(repr(API_URL))

test_url = page_url(API_URL, 1)
print("\nURL halaman 1:")
print(test_url)

response = requests.get(test_url)
print("\nStatus:", response.status_code)
print("Respons:", response.text[:300])

API_URL lengkap:
'https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_page_{page}.json'

URL halaman 1:
https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_page_1.json



Status: 200
Respons: {
 "page": 1,
 "page_size": 100,
 "total": 304,
 "has_next": true,
 "data": [
  {
   "transaction_id": "TRX00001",
   "transaction_date": "2026-01-15",
   "customer_id": "C040",
   "city": "SURABAYA",
   "product_id": "P07",
   "product_name": "Upgrade Bandwidth",
   "category": "Layanan",
   "quant


In [180]:
print("API_Q2 =", API_Q2)
print("API_CUSTOMERS =", API_CUSTOMERS)

response = requests.get(API_Q2)
print("Status transaksi:", response.status_code)
print("URL akhir:", response.url)
print("Respons:", response.text[:300])

API_Q2 = https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_q2_page_{page}.json
API_CUSTOMERS = https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/customers.json
Status transaksi: 404
URL akhir: https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_q2_page_%7Bpage%7D.json
Respons: 404: Not Found


In [181]:
# Periksa variabel dan fungsi yang sudah tersedia
print([x for x in globals() if not x.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'os', 'load_dotenv', 'find_dotenv', 'GITHUB_USER', 'REPO', 'RAW_BASE', 'API_URL', 'DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD', 'SCHEMA', 'requests', 'pd', 'create_engine', 'text', 'Date', 'URL', 'page_url', 'engine', 'conn', 'run_sql', 'API_Q2', 'API_CUSTOMERS', 'sales_data', 'next_url', 'response', 'inspect', 'test_url', 'page', 'url', 'result', 'df_sales_raw', 'df_cust_raw', 'df_sales', 'df_cust', 'df_enriched', 'col', 'converted', 'sales_check', 'cek', 'a', 'b', 'c', 'customer_data', 'px', 'revenue_segment', 'fig1', 'revenue_bulan', 'fig2', 'top5', 'fig3', 'df_q2', 'df_sales_check', 'sales_q2_data', 'df_q2_check', 'CSV_Q2', 'tanggal', 'is_iso', 'parsed', 'nama_bulan', 'urutan_bulan']


## Tahap 2 — Transform
- Transaksi: aturan yang sama dengan Exercise 2
- Pelanggan: rapikan nama, standarkan `segment` menjadi `Retail`/`SME`/`Corporate`, konversi `join_date`, hapus duplikat `customer_id`
- Gabungkan (`merge`) transaksi dengan pelanggan → `df_enriched`

In [182]:
# Salin data mentah
df_sales = df_sales_raw.copy()

# Hapus duplikat
df_sales = df_sales.drop_duplicates().copy()

# Konversi tanggal
df_sales["transaction_date"] = pd.to_datetime(
    df_sales["transaction_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Rapikan kota
df_sales["city"] = (
    df_sales["city"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Isi quantity kosong dengan 1
df_sales["quantity"] = (
    pd.to_numeric(df_sales["quantity"], errors="coerce")
    .fillna(1)
    .astype(int)
)

# Jadikan harga positif
df_sales["unit_price"] = pd.to_numeric(
    df_sales["unit_price"], errors="coerce"
).abs()

# Hitung total
df_sales["total_amount"] = (
    df_sales["quantity"] * df_sales["unit_price"]
)

In [183]:
print("Jumlah data mentah:", len(df_sales_raw))
print("Kolom:", df_sales_raw.columns.tolist())

df_sales_raw["transaction_date"] = pd.to_datetime(
    df_sales_raw["transaction_date"],
    errors="coerce"
)

print("\nRentang tanggal:")
print(df_sales_raw["transaction_date"].min())
print(df_sales_raw["transaction_date"].max())

print("\nJumlah transaksi per bulan:")
print(
    df_sales_raw["transaction_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print("\nJumlah transaction_id duplikat:")
print(df_sales_raw["transaction_id"].duplicated().sum())

Jumlah data mentah: 264
Kolom: ['transaction_id', 'transaction_date', 'customer_id', 'city', 'product_id', 'product_name', 'category', 'quantity', 'unit_price']

Rentang tanggal:
2026-04-04 00:00:00
2026-06-29 00:00:00

Jumlah transaksi per bulan:
transaction_date
2026-04    50
2026-05    91
2026-06    95
Freq: M, Name: count, dtype: int64

Jumlah transaction_id duplikat:
4


In [184]:
# Periksa struktur dan kualitas data transaksi
print("Kolom:", df_sales_raw.columns.tolist())
print("\nTipe data:")
print(df_sales_raw.dtypes)

print("\nJumlah nilai kosong:")
print(df_sales_raw.isna().sum())

print("\nJumlah duplikat seluruh baris:")
print(df_sales_raw.duplicated().sum())

print("\nData transaksi dengan ID duplikat:")
display(
    df_sales_raw[
        df_sales_raw["transaction_id"].duplicated(keep=False)
    ].sort_values("transaction_id")
)

print("\nStatistik numerik:")
display(df_sales_raw.describe(include="all").T)

Kolom: ['transaction_id', 'transaction_date', 'customer_id', 'city', 'product_id', 'product_name', 'category', 'quantity', 'unit_price']

Tipe data:
transaction_id                 str
transaction_date    datetime64[us]
customer_id                    str
city                           str
product_id                     str
product_name                   str
category                       str
quantity                       str
unit_price                     str
dtype: object

Jumlah nilai kosong:
transaction_id       0
transaction_date    28
customer_id          0
city                 0
product_id           0
product_name         0
category             0
quantity             0
unit_price           0
dtype: int64

Jumlah duplikat seluruh baris:
4

Data transaksi dengan ID duplikat:


,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price
7,TRQ00008,2026-06-02,C045,Malang,P05,Access Point Mesh,Perangkat,1,850000
10,TRQ00008,2026-06-02,C045,Malang,P05,Access Point Mesh,Perangkat,1,850000
58,TRQ00058,2026-06-04,C017,sidoarjo,P01,Paket Internet 20 Mbps,Internet,1,250000
61,TRQ00058,2026-06-04,C017,sidoarjo,P01,Paket Internet 20 Mbps,Internet,1,250000
121,TRQ00120,2026-05-18,C043,Surabaya,P04,Router WiFi,Perangkat,1,450000
124,TRQ00120,2026-05-18,C043,Surabaya,P04,Router WiFi,Perangkat,1,450000
203,TRQ00201,2026-05-05,C007,Malang,P06,Biaya Instalasi,Layanan,1,150000
206,TRQ00201,2026-05-05,C007,Malang,P06,Biaya Instalasi,Layanan,1,150000



Statistik numerik:


,count,unique,top,freq,mean,min,25%,50%,75%,max
transaction_id,264,260,TRQ00008,2,NaN,NaN,NaN,NaN,NaN,NaN
transaction_date,236,NaN,NaN,NaN,2026-05-22 05:29:29.491525,2026-04-04 00:00:00,2026-05-05 00:00:00,2026-05-23 00:00:00,2026-06-11 00:00:00,2026-06-29 00:00:00
customer_id,264,60,C045,13,NaN,NaN,NaN,NaN,NaN,NaN
city,264,13,Malang,35,NaN,NaN,NaN,NaN,NaN,NaN
product_id,264,7,P01,65,NaN,NaN,NaN,NaN,NaN,NaN
product_name,264,7,Paket Internet 20 Mbps,65,NaN,NaN,NaN,NaN,NaN,NaN
category,264,3,Internet,156,NaN,NaN,NaN,NaN,NaN,NaN
quantity,264,4,1,180,NaN,NaN,NaN,NaN,NaN,NaN
unit_price,264,11,250000,65,NaN,NaN,NaN,NaN,NaN,NaN


In [185]:
# Periksa nilai quantity dan unit_price
for col in ["quantity", "unit_price"]:
    print(f"\n--- {col} ---")
    converted = pd.to_numeric(df_sales_raw[col], errors="coerce")
    print("Nilai gagal dikonversi:", converted.isna().sum())
    print("Nilai unik:", df_sales_raw[col].unique()[:20])

# Periksa jumlah baris setelah setiap tahap
print("\nData awal:", len(df_sales_raw))

sales_check = df_sales_raw.drop_duplicates()
print("Setelah hapus duplikat:", len(sales_check))

sales_check = sales_check.dropna(subset=["transaction_date"])
print("Setelah hapus tanggal kosong:", len(sales_check))


--- quantity ---
Nilai gagal dikonversi: 11
Nilai unik: <ArrowStringArray>
['3', '1', '2', '']
Length: 4, dtype: str

--- unit_price ---
Nilai gagal dikonversi: 0
Nilai unik: <ArrowStringArray>
[ '250000',  '150000',  '200000',  '450000',  '650000',  '850000',  '400000',
 '-400000', '-150000', '-450000', '-650000']
Length: 11, dtype: str

Data awal: 264
Setelah hapus duplikat: 260
Setelah hapus tanggal kosong: 232


In [186]:
# Hitung pengaruh setiap aturan pembersihan
cek = df_sales_raw.copy()

cek["transaction_date"] = pd.to_datetime(
    cek["transaction_date"], errors="coerce"
)
cek["quantity"] = pd.to_numeric(
    cek["quantity"], errors="coerce"
)

print("Data awal:", len(cek))
print("Duplikat seluruh baris:", cek.duplicated().sum())
print("Tanggal kosong:", cek["transaction_date"].isna().sum())
print("Quantity kosong:", cek["quantity"].isna().sum())

# Hapus duplikat saja
a = cek.drop_duplicates()
print("Setelah hapus duplikat:", len(a))

# Hapus duplikat dan tanggal kosong
b = a.dropna(subset=["transaction_date"])
print("Setelah hapus tanggal kosong:", len(b))

# Hapus duplikat, tanggal kosong, dan quantity kosong
c = b.dropna(subset=["quantity"])
print("Setelah ketiganya dibersihkan:", len(c))

Data awal: 264
Duplikat seluruh baris: 4
Tanggal kosong: 28
Quantity kosong: 11
Setelah hapus duplikat: 260
Setelah hapus tanggal kosong: 232
Setelah ketiganya dibersihkan: 222


In [187]:
cek = df_sales_raw.copy()

cek["transaction_date"] = pd.to_datetime(
    cek["transaction_date"], errors="coerce"
)
cek["quantity"] = pd.to_numeric(
    cek["quantity"], errors="coerce"
)

cek = cek.drop_duplicates()

display(
    cek.loc[
        cek["transaction_date"].notna()
        & cek["quantity"].isna()
    ]
)

,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price
23,TRQ00023,2026-04-26,C059,Malang,P05,Access Point Mesh,Perangkat,NaN,850000
46,TRQ00046,2026-05-23,C001,sidoarjo,P06,Biaya Instalasi,Layanan,NaN,150000
70,TRQ00069,2026-05-28,C050,Malang,P01,Paket Internet 20 Mbps,Internet,NaN,250000
93,TRQ00092,2026-05-04,C005,sidoarjo,P04,Router WiFi,Perangkat,NaN,450000
116,TRQ00115,2026-06-22,C058,Surabaya,P01,Paket Internet 20 Mbps,Internet,NaN,250000
140,TRQ00138,2026-05-23,C009,malang,P02,Paket Internet 50 Mbps,Internet,NaN,400000
163,TRQ00161,2026-06-02,C011,Surabaya,P01,Paket Internet 20 Mbps,Internet,NaN,250000
186,TRQ00184,2026-06-06,C031,sidoarjo,P02,Paket Internet 50 Mbps,Internet,NaN,400000
233,TRQ00230,2026-04-22,C025,Batu,P07,Upgrade Bandwidth,Layanan,NaN,200000
256,TRQ00253,2026-05-18,C055,Malang,P01,Paket Internet 20 Mbps,Internet,NaN,250000


In [188]:
print("Jumlah df_sales:", len(df_sales))
print("Jumlah df_sales_raw:", len(df_sales_raw))
print("Jumlah df_cust:", len(df_cust))
print("Jumlah df_enriched:", len(df_enriched))

Jumlah df_sales: 260
Jumlah df_sales_raw: 264
Jumlah df_cust: 60
Jumlah df_enriched: 260


In [189]:
cek = df_sales_raw.copy()

print("Data awal:", len(cek))

cek = cek.drop_duplicates()
print("Setelah hapus duplikat:", len(cek))

cek["transaction_date"] = pd.to_datetime(
    cek["transaction_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Tanggal kosong:", cek["transaction_date"].isna().sum())

cek["quantity"] = pd.to_numeric(
    cek["quantity"], errors="coerce"
)

print("Quantity kosong:", cek["quantity"].isna().sum())

print("Data dengan tanggal valid:",
      cek["transaction_date"].notna().sum())

Data awal: 264
Setelah hapus duplikat: 260
Tanggal kosong: 28
Quantity kosong: 11
Data dengan tanggal valid: 232


In [190]:
print("Jumlah transaksi:", len(df_sales))
print("Tanggal kosong:", df_sales["transaction_date"].isna().sum())
print("Jumlah pelanggan:", len(df_cust))

Jumlah transaksi: 260
Tanggal kosong: 0
Jumlah pelanggan: 60


In [191]:
# Periksa tanggal kosong berdasarkan kota
display(
    df_sales.loc[
        df_sales["transaction_date"].isna()
    ].groupby("city").size()
)

# Periksa jumlah tanggal kosong dan duplikat ID transaksi
print("Tanggal kosong:", df_sales["transaction_date"].isna().sum())
print(
    "ID transaksi duplikat:",
    df_sales["transaction_id"].duplicated().sum()
)

Series([], dtype: int64)

Tanggal kosong: 0
ID transaksi duplikat: 0


In [192]:
# Periksa distribusi data berdasarkan kota
print("Jumlah transaksi per kota:")
display(df_sales["city"].value_counts(dropna=False))

# Periksa apakah ada tanggal di luar periode Q2
print("\nTanggal paling awal:", df_sales["transaction_date"].min())
print("Tanggal paling akhir:", df_sales["transaction_date"].max())

# Periksa data tanggal kosong
display(
    df_sales.loc[
        df_sales["transaction_date"].isna()
    ].head(10)
)

Jumlah transaksi per kota:


city
Malang      118
Surabaya     84
Sidoarjo     39
Batu         14
Pasuruan      5
Name: count, dtype: int64[pyarrow]


Tanggal paling awal: 2026-01-05 00:00:00
Tanggal paling akhir: 2026-12-06 00:00:00


,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price,total_amount


In [193]:
# Cari variabel yang berhubungan dengan validasi Q2
[x for x in globals() if any(
    kata in x.lower()
    for kata in ["assert", "q2", "clean", "sales", "expected"]
)]

['API_Q2',
 'sales_data',
 'df_sales_raw',
 'df_sales',
 'sales_check',
 'df_q2',
 'df_sales_check',
 'sales_q2_data',
 'df_q2_check',
 'CSV_Q2']

In [194]:
# Perbaiki URL sumber transaksi Q2
API_Q2 = API_URL

# Tes koneksi
response = requests.get(page_url(API_Q2, 1))
print("Status:", response.status_code)

Status: 200


In [195]:
print("df_sales:", len(df_sales))
print("df_cust:", len(df_cust))
print("df_enriched:", len(df_enriched))

df_sales: 260
df_cust: 60
df_enriched: 260


In [196]:
# Periksa jumlah baris untuk setiap tahap
cek = df_sales_raw.drop_duplicates().copy()

cek["transaction_date"] = pd.to_datetime(
    cek["transaction_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Setelah duplikat dihapus:", len(cek))
print("Tanggal valid:", cek["transaction_date"].notna().sum())
print("Tanggal kosong:", cek["transaction_date"].isna().sum())

Setelah duplikat dihapus: 260
Tanggal valid: 232
Tanggal kosong: 28


In [197]:
# Pulihkan URL khusus transaksi Q2
API_Q2 = f"{RAW_BASE}/api/static/sales_q2_page_{{page}}.json"

# Tes halaman pertama Q2
url = page_url(API_Q2, 1)
response = requests.get(url)

print("URL:", url)
print("Status:", response.status_code)

if response.status_code == 200:
    print(response.json().keys())
else:
    print(response.text[:200])

URL: https://raw.githubusercontent.com/AisahAlfiyatusR/data-science-workshop/main/api/static/sales_q2_page_1.json
Status: 200
dict_keys(['page', 'page_size', 'total', 'has_next', 'data'])


In [198]:
# Extract semua halaman transaksi Q2
sales_data = []
page = 1

while True:
    response = requests.get(page_url(API_Q2, page))
    response.raise_for_status()
    result = response.json()

    sales_data.extend(result["data"])

    if not result.get("has_next", False):
        break

    page += 1

df_sales_raw = pd.DataFrame(sales_data)

print("Total data transaksi Q2:", len(df_sales_raw))
display(df_sales_raw.head())

Total data transaksi Q2: 264


,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price
0,TRQ00001,2026-04-20,C058,surabaya,P01,Paket Internet 20 Mbps,Internet,3,250000
1,TRQ00002,2026-05-16,C015,sidoarjo,P06,Biaya Instalasi,Layanan,1,150000
2,TRQ00003,2026-06-18,C037,Sidoarjo,P01,Paket Internet 20 Mbps,Internet,3,250000
3,TRQ00004,2026-06-17,C012,Malang,P06,Biaya Instalasi,Layanan,1,150000
4,TRQ00005,2026-06-26,C046,surabaya,P07,Upgrade Bandwidth,Layanan,1,200000


In [199]:
print("Total data mentah:", len(df_sales_raw))
print("Jumlah baris duplikat:", df_sales_raw.duplicated().sum())

Total data mentah: 264
Jumlah baris duplikat: 4


In [200]:
# Transform data transaksi Q2
df_sales = df_sales_raw.drop_duplicates().copy()

df_sales["transaction_date"] = pd.to_datetime(
    df_sales["transaction_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

df_sales["city"] = df_sales["city"].astype(str).str.strip().str.title()

df_sales["quantity"] = pd.to_numeric(
    df_sales["quantity"], errors="coerce"
).fillna(1).astype(int)

df_sales["unit_price"] = pd.to_numeric(
    df_sales["unit_price"], errors="coerce"
).abs()

df_sales["total_amount"] = df_sales["quantity"] * df_sales["unit_price"]

print("Jumlah transaksi setelah dibersihkan:", len(df_sales))
print("Tanggal kosong:", df_sales["transaction_date"].isna().sum())
display(df_sales.head())

Jumlah transaksi setelah dibersihkan: 260
Tanggal kosong: 0


,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price,total_amount
0,TRQ00001,2026-04-20,C058,Surabaya,P01,Paket Internet 20 Mbps,Internet,3,250000,750000
1,TRQ00002,2026-05-16,C015,Sidoarjo,P06,Biaya Instalasi,Layanan,1,150000,150000
2,TRQ00003,2026-06-18,C037,Sidoarjo,P01,Paket Internet 20 Mbps,Internet,3,250000,750000
3,TRQ00004,2026-06-17,C012,Malang,P06,Biaya Instalasi,Layanan,1,150000,150000
4,TRQ00005,2026-06-26,C046,Surabaya,P07,Upgrade Bandwidth,Layanan,1,200000,200000


In [201]:
# Extract data pelanggan
response = requests.get(API_CUSTOMERS)
response.raise_for_status()

customer_data = response.json()
df_cust_raw = pd.DataFrame(customer_data["data"])

print("Total data pelanggan mentah:", len(df_cust_raw))
display(df_cust_raw.head())

Total data pelanggan mentah: 64


,customer_id,customer_name,segment,join_date
0,C001,PT Mitra Digital,SME,2025-11-28
1,C002,Fajar Kartika,Retail,2025-11-17
2,C003,Eko Putri,Retail,2025-02-13
3,C004,PT Maju Mandiri,sme,2025-03-18
4,C005,Budi Fajar,Retail,2025-10-10


In [202]:
print(df_cust_raw.columns.tolist())
print(df_cust_raw.head(2))

['customer_id', 'customer_name', 'segment', 'join_date']
  customer_id     customer_name  segment   join_date
0        C001  PT Mitra Digital      SME  2025-11-28
1        C002     Fajar Kartika  Retail   2025-11-17


In [203]:
# Transform data pelanggan
df_cust = df_cust_raw.copy()

df_cust["customer_name"] = (
    df_cust["customer_name"].astype(str).str.strip().str.title()
)

df_cust["segment"] = df_cust["segment"].astype(str).str.strip().str.title()

df_cust["join_date"] = pd.to_datetime(
    df_cust["join_date"], errors="coerce"
)

df_cust = df_cust.drop_duplicates(subset=["customer_id"]).copy()

print("Jumlah pelanggan unik:", len(df_cust))
print("ID pelanggan unik:", df_cust["customer_id"].is_unique)
print("Kategori segmen:", set(df_cust["segment"]))
display(df_cust.head())

Jumlah pelanggan unik: 60
ID pelanggan unik: True
Kategori segmen: {'Retail', 'Sme', 'Corporate'}


,customer_id,customer_name,segment,join_date
0,C001,Pt Mitra Digital,Sme,2025-11-28
1,C002,Fajar Kartika,Retail,2025-11-17
2,C003,Eko Putri,Retail,2025-02-13
3,C004,Pt Maju Mandiri,Sme,2025-03-18
4,C005,Budi Fajar,Retail,2025-10-10


In [204]:
# Gabungkan transaksi dengan data pelanggan
df_enriched = df_sales.merge(
    df_cust,
    on="customer_id",
    how="left"
)

print("Jumlah transaksi:", len(df_sales))
print("Jumlah data setelah merge:", len(df_enriched))
print("Segmen yang kosong:", df_enriched["segment"].isna().sum())

display(df_enriched.head())

Jumlah transaksi: 260
Jumlah data setelah merge: 260
Segmen yang kosong: 0


,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price,total_amount,customer_name,segment,join_date
0,TRQ00001,2026-04-20,C058,Surabaya,P01,Paket Internet 20 Mbps,Internet,3,250000,750000,Nanda Vina,Retail,2025-12-20
1,TRQ00002,2026-05-16,C015,Sidoarjo,P06,Biaya Instalasi,Layanan,1,150000,150000,Pt Sinar Abadi,Corporate,2025-03-22
2,TRQ00003,2026-06-18,C037,Sidoarjo,P01,Paket Internet 20 Mbps,Internet,3,250000,750000,Pt Cahaya Mandiri,Sme,2025-05-19
3,TRQ00004,2026-06-17,C012,Malang,P06,Biaya Instalasi,Layanan,1,150000,150000,Pt Sinar Digital,Sme,2025-09-16
4,TRQ00005,2026-06-26,C046,Surabaya,P07,Upgrade Bandwidth,Layanan,1,200000,200000,Pt Nusantara Mandiri,Sme,2025-05-14


In [205]:
df_cust["segment"] = df_cust["segment"].replace({
    "Sme": "SME",
    "sme": "SME",
    "sME": "SME"
})

# Perbarui hasil merge agar mengikuti segment yang sudah diperbaiki
df_enriched = df_sales.merge(
    df_cust,
    on="customer_id",
    how="left"
)

print(df_cust["segment"].unique())

<ArrowStringArray>
['SME', 'Retail', 'Corporate']
Length: 3, dtype: str


In [206]:
assert len(df_sales) == 260, "transaksi Q2 bersih harus 260 baris"
assert df_cust["customer_id"].is_unique and len(df_cust) == 60, "pelanggan unik harus 60"
assert set(df_cust["segment"]) == {"Retail", "SME", "Corporate"}
assert len(df_enriched) == 260 and df_enriched["segment"].notna().all()
print("✅ Transform lolos")

✅ Transform lolos


## Tahap 3 — Load
Muat `df_enriched` ke tabel `sales_q2_enriched` di schema Anda.

In [207]:
run_sql("""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name NOT IN ('pg_catalog', 'information_schema')
ORDER BY schema_name;
""")

,schema_name
0,peserta25
1,public
2,ref


In [208]:
run_sql("""
SELECT
    has_schema_privilege(current_user, 'peserta25', 'USAGE') AS can_use,
    has_schema_privilege(current_user, 'peserta25', 'CREATE') AS can_create;
""")

,can_use,can_create
0,True,True


In [209]:
df_enriched.to_sql(
    name="sales_q2_enriched",
    con=engine,
    schema="peserta25",
    if_exists="replace",
    index=False
)

print("✅ Data berhasil dimuat!")
print("Jumlah baris:", len(df_enriched))

✅ Data berhasil dimuat!
Jumlah baris: 260


## Tahap 4 — SQL
Jawab dengan query: (a) revenue & jumlah transaksi per segmen, (b) revenue per segmen per bulan, (c) top 5 pelanggan beserta nama & segmennya.

In [210]:
run_sql("""
SELECT
    segment,
    SUM(total_amount) AS revenue,
    COUNT(*) AS jumlah_transaksi
FROM peserta25.sales_q2_enriched
GROUP BY segment
ORDER BY revenue DESC;
""")

run_sql("""
SELECT
    segment,
    DATE_TRUNC('month', transaction_date)::date AS bulan,
    SUM(total_amount) AS revenue
FROM peserta25.sales_q2_enriched
GROUP BY
    segment,
    DATE_TRUNC('month', transaction_date)::date
ORDER BY
    bulan,
    segment;
""")

run_sql("""
SELECT
    customer_id,
    customer_name,
    segment,
    SUM(total_amount) AS total_revenue
FROM peserta25.sales_q2_enriched
GROUP BY
    customer_id,
    customer_name,
    segment
ORDER BY total_revenue DESC
LIMIT 5;
""")

,customer_id,customer_name,segment,total_revenue
0,C045,Pt Mitra Digital,SME,5900000.0
1,C009,Teguh Teguh,Retail,5250000.0
2,C013,Pt Berkah Sentosa,SME,5250000.0
3,C011,Pt Berkah Sentosa,SME,4300000.0
4,C050,Gita Joko,Retail,4300000.0


## Tahap 5 — Dashboard
Buat minimal 3 visual (Plotly atau Looker Studio).

In [211]:

import pandas as pd
import plotly.express as px

# Pastikan tanggal sudah terbaca sebagai datetime
df_enriched["transaction_date"] = pd.to_datetime(
    df_enriched["transaction_date"], errors="coerce"
)

# Filter transaksi Q2 2026: April sampai Juni
df_q2 = df_enriched[
    (df_enriched["transaction_date"] >= "2026-04-01") &
    (df_enriched["transaction_date"] < "2026-07-01")
].copy()

print("Jumlah transaksi Q2:", len(df_q2))
print(
    "Periode:",
    df_q2["transaction_date"].min(),
    "sampai",
    df_q2["transaction_date"].max()
)

# ==========================================
# VISUAL 1: TOTAL REVENUE PER SEGMEN
# ==========================================
revenue_segment = (
    df_q2.groupby("segment", as_index=False)["total_amount"]
    .sum()
    .sort_values("total_amount", ascending=False)
)

fig1 = px.bar(
    revenue_segment,
    x="segment",
    y="total_amount",
    title="Total Revenue per Segmen — Q2 2026",
    labels={
        "segment": "Segmen Pelanggan",
        "total_amount": "Revenue (Rp)"
    },
    text_auto=".3s"
)

fig1.update_layout(
    xaxis_title="Segmen Pelanggan",
    yaxis_title="Revenue (Rp)"
)
fig1.show()


# ==========================================
# VISUAL 2: TREN REVENUE PER BULAN
# ==========================================
revenue_bulan = (
    df_q2.assign(bulan=df_q2["transaction_date"].dt.month)
    .groupby("bulan", as_index=False)["total_amount"]
    .sum()
)

nama_bulan = {
    4: "April 2026",
    5: "Mei 2026",
    6: "Juni 2026"
}

revenue_bulan["bulan"] = revenue_bulan["bulan"].map(nama_bulan)

urutan_bulan = ["April 2026", "Mei 2026", "Juni 2026"]
revenue_bulan["bulan"] = pd.Categorical(
    revenue_bulan["bulan"],
    categories=urutan_bulan,
    ordered=True
)
revenue_bulan = revenue_bulan.sort_values("bulan")

fig2 = px.line(
    revenue_bulan,
    x="bulan",
    y="total_amount",
    markers=True,
    title="Tren Revenue Q2 2026 per Bulan",
    labels={
        "bulan": "Bulan",
        "total_amount": "Revenue (Rp)"
    }
)

fig2.update_layout(
    xaxis_title="Bulan",
    yaxis_title="Revenue (Rp)"
)
fig2.show()


# ==========================================
# VISUAL 3: TOP 5 PELANGGAN
# ==========================================
top5 = (
    df_q2.groupby(
        ["customer_id", "customer_name", "segment"],
        as_index=False
    )["total_amount"]
    .sum()
    .sort_values("total_amount", ascending=False)
    .head(5)
    .sort_values("total_amount", ascending=True)
)

fig3 = px.bar(
    top5,
    x="total_amount",
    y="customer_name",
    color="segment",
    orientation="h",
    title="Top 5 Pelanggan berdasarkan Revenue — Q2 2026",
    labels={
        "total_amount": "Revenue (Rp)",
        "customer_name": "Nama Pelanggan",
        "segment": "Segmen"
    },
    text_auto=".3s"
)

fig3.update_layout(
    xaxis_title="Revenue (Rp)",
    yaxis_title="Nama Pelanggan"
)
fig3.show()


Jumlah transaksi Q2: 197
Periode: 2026-04-04 00:00:00 sampai 2026-06-30 00:00:00


## Tahap 6 — Insight & Rekomendasi
Tuliskan 3 temuan utama dan 1 rekomendasi bisnis.

1. ...
2. ...
3. ...

**Rekomendasi:** ...

### 3 Temuan Utama

1. **Segmen Retail menjadi penyumbang revenue terbesar.** Pada Q2 2026, segmen Retail menghasilkan revenue sebesar Rp60,5 juta, diikuti SME sebesar Rp28,7 juta dan Corporate sebesar Rp10,5 juta. Hal ini menunjukkan bahwa Retail merupakan segmen yang memberikan kontribusi pendapatan paling besar bagi perusahaan.

2. **Revenue meningkat secara konsisten selama Q2 2026.** Revenue naik dari sekitar Rp28,0 juta pada April menjadi Rp33,4 juta pada Mei dan Rp38,1 juta pada Juni. Peningkatan ini menunjukkan tren penjualan yang positif selama kuartal kedua 2026.

3. **Teguh Teguh menjadi pelanggan dengan revenue tertinggi.** Pelanggan ini menghasilkan revenue sebesar Rp5,25 juta, diikuti Gita Joko sebesar Rp4,30 juta dan PT Berkah Sentosa sebesar Rp4,05 juta. Selain itu, PT Berkah Mandiri dan PT Mitra Digital masing-masing menghasilkan revenue sebesar Rp3,40 juta.

**Rekomendasi:** Perusahaan sebaiknya mempertahankan pelanggan pada segmen Retail melalui program loyalitas dan penawaran khusus karena segmen ini memberikan kontribusi revenue terbesar. Perusahaan juga perlu menjaga hubungan dengan pelanggan utama, terutama Teguh Teguh, Gita Joko, dan PT Berkah Sentosa. Selain itu, strategi pemasaran pada segmen SME dan Corporate perlu ditingkatkan agar kontribusi kedua segmen tersebut terhadap total revenue dapat bertambah. Tren peningkatan revenue dari April hingga Juni juga dapat dijadikan pertimbangan dalam menetapkan target penjualan untuk kuartal berikutnya.

## 📤 Pengumpulan
1. Simpan notebook (`Ctrl+S`), lalu di panel **Source Control** VS Code: stage → commit → **Sync Changes/Push** ke fork Anda
2. Isi juga `submissions/NAMA_KELOMPOK.md` di fork Anda (template di folder `submissions/`)
3. Kirim link fork Anda ke form pengumpulan dari panitia